# Step 7 — Configure QLoRA

Prepare the quantized Qwen2.5-VL model for low-bit training using `prepare_model_for_kbit_training()`, then attach LoRA adapters with PEFT. Freeze the vision encoder so only the language model is adapted, target the attention projection layers (`q_proj` and `v_proj`), and verify the number of trainable parameters. This ensures that only lightweight LoRA adapters are updated during fine-tuning.

In [ ]:
# Import PEFT utilities
from peft import LoraConfig,get_peft_model,prepare_model_for_kbit_training

# Prepare model for k-bit training
model=prepare_model_for_kbit_training(model)

# Freeze all vision encoder parameters
for name,param in model.named_parameters():
    if name.startswith("model.visual."):
        param.requires_grad=False

vision_trainable=sum(
    p.numel()
    for n,p in model.named_parameters()
    if n.startswith("model.visual.") and p.requires_grad
)

print(f"Vision trainable parameters: {vision_trainable}")

# Configure LoRA
lora_config=LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "v_proj"
    ]
)

# Attach LoRA adapters
model=get_peft_model(model,lora_config)

# Print trainable parameter statistics
model.print_trainable_parameters()

# Verify vision encoder is frozen
vision_trainable=sum(
    p.numel()
    for n,p in model.named_parameters()
    if n.startswith("model.visual.") and p.requires_grad
)

language_trainable=sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(f"Vision Trainable Parameters : {vision_trainable:,}")
print(f"Total Trainable Parameters : {language_trainable:,}")

Vision trainable parameters: 0
trainable params: 1,843,200 || all params: 3,756,466,176 || trainable%: 0.0491
Vision Trainable Parameters : 0
Total Trainable Parameters : 1,843,200


# Step 8 — Sanity Check

Before starting full fine-tuning, perform a single forward and backward pass using one mini-batch from the training set. This verifies that the dataset, processor, data collator, QLoRA model, and optimizer are correctly connected. Confirm that the loss is finite, gradients are computed, GPU memory remains stable, and no runtime errors or out-of-memory issues occur before launching long training.

In [ ]:
# Import required libraries
from torch.utils.data import DataLoader
import torch

# Create training dataloader
train_loader=DataLoader(
    train_dataset,
    batch_size=1,
    shuffle=True,
    collate_fn=data_collator
)

# Create optimizer
optimizer=torch.optim.AdamW(
    model.parameters(),
    lr=2e-4
)

# Enable training mode
model.train()
model.gradient_checkpointing_enable()

# Get one training batch
batch=next(iter(train_loader))

# Move tensors to GPU
batch={k:v.to(model.device) if isinstance(v,torch.Tensor) else v for k,v in batch.items()}

# Clear previous gradients
optimizer.zero_grad()

# Forward pass
outputs=model(**batch)

# Compute loss
loss=outputs.loss

# Verify loss
assert torch.isfinite(loss),"Loss is NaN or Inf."

print(f"Initial Loss : {loss.item():.4f}")

# Backward pass
loss.backward()

# Verify gradients
grad_found=False

for name,param in model.named_parameters():
    if param.requires_grad and param.grad is not None:
        grad_found=True
        break

assert grad_found,"No gradients were computed."

print("Backward pass successful.")

# Optimizer step
optimizer.step()

# Clear gradients
optimizer.zero_grad()

# Display GPU memory usage
allocated=torch.cuda.memory_allocated()/1024**3
reserved=torch.cuda.memory_reserved()/1024**3

print(f"GPU Memory Allocated : {allocated:.2f} GB")
print(f"GPU Memory Reserved : {reserved:.2f} GB")

print("Sanity check completed successfully.")

[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Initial Loss : 6.0331
Backward pass successful.
GPU Memory Allocated : 3.06 GB
GPU Memory Reserved : 3.89 GB
Sanity check completed successfully.


# Step 9 — Fine-tune

Configure the Hugging Face `Trainer` using `TrainingArguments` optimized for QLoRA fine-tuning. Enable gradient accumulation, gradient checkpointing, FP16 mixed precision, cosine learning-rate scheduling, checkpoint saving, and early stopping. Evaluate **only on the validation set** after each epoch, automatically load the best checkpoint based on validation loss, and never use the test set during training or model selection.

In [ ]:
# Import required libraries
from transformers import TrainingArguments,Trainer,EarlyStoppingCallback

print("train args start")

# Configure training arguments
training_args=TrainingArguments(
    output_dir="./qwen2.5vl_lora_output",
    num_train_epochs=5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=50,
    weight_decay=0.01,
    fp16=True,
    gradient_checkpointing=True,
    do_train=True,
    do_eval=True,
    use_cache=False,
    logging_strategy="steps",
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    remove_unused_columns=False,
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
    max_grad_norm=1.0
)

model.config.use_cache=False
model.enable_input_require_grads()

print("train args end")

train args start
train args end


In [ ]:
print("trainer start")
# Create trainer
trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    processing_class=processor,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=2
        )
    ]
)

print(len(train_dataset),len(val_dataset))
batch=data_collator([train_dataset[0]])
print(batch.keys())

print("trainer end")

trainer start
1386 315
KeysView({'input_ids': tensor([[151644,   8948,    198,   2610,    525,    458,   6203,    389,   2688,
            971,     68,  17795,  23316,     13, 151645,    198, 151644,    872,
            198, 151652, 151655, 151655, 151655, 151655, 151655, 151655, 151655,
         151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655,
         151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655,
         151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655,
         151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655,
         151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655,
         151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655,
         151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655,
         151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655,
         151655, 151655, 151655, 151655, 151655, 151655, 151655

In [ ]:
print("training start")
# Start fine-tuning
trainer.train()
print("training end")

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 151645, 'bos_token_id': None, 'pad_token_id': 151643}.


training start


Epoch,Training Loss,Validation Loss
1,1.539243,1.136842
2,0.661225,0.819064
3,0.679032,0.846797
4,0.756808,0.872887


training end


In [ ]:
print("eval best checkpoint start")

# Evaluate best checkpoint on validation set
validation_metrics=trainer.evaluate()

# Display validation metrics
print(validation_metrics)

print("eval best checkpoint end")

eval best checkpoint start


Training Loss,Validation Loss,Epoch
0.756808,0.819064,4


{'eval_loss': 0.8190642595291138}
eval best checkpoint end


# Step 10 — Save Model

After fine-tuning, save the trained **LoRA adapters**, processor, tokenizer, training arguments, and training logs. These files are required to reload the fine-tuned model for inference without retraining. Since Kaggle sessions are temporary, immediately save the artifacts to the Kaggle Output directory and optionally upload them to Hugging Face Hub or GitHub for permanent storage.

In [ ]:
# Import required libraries
import os
import json
import torch
import shutil
from transformers import TrainerState

# Create output directory
SAVE_DIR="/kaggle/working/qwen2.5vl_lora"
os.makedirs(SAVE_DIR,exist_ok=True)

# Save LoRA adapter
model.save_pretrained(os.path.join(SAVE_DIR,"adapter"))

# Save processor
processor.save_pretrained(os.path.join(SAVE_DIR,"processor"))

# Save tokenizer
processor.tokenizer.save_pretrained(os.path.join(SAVE_DIR,"tokenizer"))

# Save training arguments
torch.save(
    trainer.args,
    os.path.join(SAVE_DIR, "training_args.pt")
)

# Save trainer state
trainer.state.save_to_json(os.path.join(SAVE_DIR,"trainer_state.json"))

# Save evaluation metrics
validation_metrics=trainer.evaluate()

with open(os.path.join(SAVE_DIR,"validation_metrics.json"),"w") as f:
    json.dump(validation_metrics,f,indent=4)

# Save training log history
with open(os.path.join(SAVE_DIR,"log_history.json"),"w") as f:
    json.dump(trainer.state.log_history,f,indent=4)

# Create compressed backup
shutil.make_archive(
    base_name="/kaggle/working/qwen2.5vl_lora_backup",
    format="zip",
    root_dir=SAVE_DIR
)

print(f"Model saved to: {SAVE_DIR}")
print("Backup ZIP created successfully.")

Training Loss,Validation Loss,Epoch
0.756808,0.819064,4


Model saved to: /kaggle/working/qwen2.5vl_lora
Backup ZIP created successfully.
